# Test Fine-tuning Results
Test the fine-tuned LIANet Creoss region performance to the local model perfomance

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import glob
from rasterio.windows import Window
from utils import s2_to_rgb, _preprocess_S2
# Add src to path
sys.path.insert(0, '/home/user/src')
import rasterio as rio
from datasets import PASTIS
from models.models_finetune import UNet, MicroUNet
from utils import _preprocess_S2
from settings import *
from datetime import datetime

from helpers import load_model_class


from torchmetrics import MetricCollection

from metrics import multiclass_segmentation_metrics
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

pretrained_model_path = {"PASTIS_joint_T31TFM": "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01",
                         "PASTIS_joint_T32ULU": "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01",
                         "PASTIS_joint_T31TFJ": "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01",
                         "PASTIS_joint_T30UXV": "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01",
                         "PASTIS_local_T31TFM": "/home/user/results_shared/fourier_learned_T31TFM/2026-04-08_00-13-56",
                         "PASTIS_local_T32ULU": "/home/user/results_shared/fourier_learned_T32ULU/2026-04-04_09-21-54",
                         "PASTIS_local_T31TFJ": "/home/user/results_shared/fourier_learned_T31TFJ/2026-04-02_13-40-05",
                         "PASTIS_local_T30UXV": "/home/user/results_shared/fourier_learned_T30UXV/2026-04-06_04-50-55",}



def load_model(CKPT_PATH, other_task, model_type):
    model_finetune = load_model_class(
            other_task, 
            model_type, 
            MODEL_PATH= pretrained_model_path[other_task],
            NUM_CLASSES=num_classes[other_task],
            ACTIVATION_FUNCTION=activation_functions[other_task]
        )
    # model_finetune = DownstreamModel(
    #     model_path=pretrained_model_path,
    #     checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
    #     adaption_strategy="replace_final_block",
    #     num_classes=num_classes[other_task],
    #     activation="none"
    # )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune

In [3]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import multiclass_segmentation_metrics

# model_type_list = ["replace_final_block", "unet", "micro_unet"]
model_type_list = ["unet"]

for model_type in model_type_list:
    if model_type == "replace_final_block":
        all_tasks_list = {
            "PASTIS_joint_T31TFM":["PASTIS_joint_T31TFJ", "PASTIS_joint_T32ULU", "PASTIS_joint_T30UXV"],
            "PASTIS_joint_T31TFJ":["PASTIS_joint_T31TFM", "PASTIS_joint_T32ULU", "PASTIS_joint_T30UXV"],
            "PASTIS_joint_T32ULU":["PASTIS_joint_T31TFM", "PASTIS_joint_T31TFJ", "PASTIS_joint_T30UXV"],
            "PASTIS_joint_T30UXV":["PASTIS_joint_T31TFM", "PASTIS_joint_T31TFJ", "PASTIS_joint_T32ULU"],
            "PASTIS_local_T31TFM":["PASTIS_local_T31TFM"],
            "PASTIS_local_T32ULU":["PASTIS_local_T32ULU"],
            "PASTIS_local_T31TFJ":["PASTIS_local_T31TFJ"],
            "PASTIS_local_T30UXV":["PASTIS_local_T30UXV"],
        }
    elif model_type == "unet" or model_type == "micro_unet":
        all_tasks_list = {
            "PASTIS_local_T31TFM":["PASTIS_local_T32ULU", "PASTIS_local_T31TFJ", "PASTIS_local_T30UXV"],
            "PASTIS_local_T32ULU":["PASTIS_local_T31TFM", "PASTIS_local_T31TFJ", "PASTIS_local_T30UXV"],
            "PASTIS_local_T31TFJ":["PASTIS_local_T31TFM", "PASTIS_local_T32ULU", "PASTIS_local_T30UXV"],
            "PASTIS_local_T30UXV":["PASTIS_local_T31TFM", "PASTIS_local_T32ULU", "PASTIS_local_T31TFJ"],
        }

    VAL_FOLDS = [1, 2, 3, 4, 5]
    BATCH_SIZE = 16
    NUM_WORKERS = 8

    results_rows = []


    for Target_region in all_tasks_list:
        application_name = Target_region.split("_")[0]
        other_tasks = all_tasks_list[Target_region]

        for val_fold in VAL_FOLDS:
            val_dataset = PASTIS(
                top_dir=TOP_DIR[Target_region],
                s2_tiles=s2_tiles[Target_region],
                labels=labels[Target_region],
                train_val_key="val",
                val_folds=[val_fold],
            )

            dataloader = torch.utils.data.DataLoader(
                val_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                drop_last=False,
            )

            for source_region in other_tasks:
                startername = "LIANet" if "replace_final_block" in model_type else "micro_unet" if "micro_unet" in model_type else "unet"
                model_dir = glob.glob(f"/home/user/results_local/finetuning_results_PASTIS/{source_region}/{startername}_valFolds{val_fold}*")
                # print(model_dir)
                # model_dir = (
                #     f"/home/user/results_local/finetuning_results/"
                #     f"{source_region}/LIANet_valFolds{val_fold}_lr0.0001_batchsize16"
                # )

                for run_name in sorted(os.listdir(model_dir[0])):
                    ckpt_path = os.path.join(model_dir[0], run_name, "last.pt")

                    if not os.path.exists(ckpt_path):
                        continue

                    model = load_model(ckpt_path, source_region, model_type)
                    model.eval()

                    list_of_metrics, _ = multiclass_segmentation_metrics(
                        num_classes=num_classes[Target_region],
                        ignore_index=255,
                    )

                    metrictracker = MetricCollection(list_of_metrics).to(device)

                    with torch.no_grad():
                        for batch in tqdm(
                            dataloader,
                            desc=f"{Target_region} | fold {val_fold} | {source_region} | {run_name}",
                            leave=False,
                        ):
                            x = batch["x_s2"].to(device)
                            y = batch["y_s2"].to(device)
                            label = batch["label"].to(device)
                            delta_days = batch["delta_days"].to(device)
                            target_tile = Target_region.split("_")[-1]
                            s2data  = batch["s2data"].to(device)

                            if "local" in Target_region:
                                region_idx = 0
                            elif "BFP" in Target_region:
                                region_idx = 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else None
                            elif "joint" and "PASTIS" in Target_region:
                                region_indx = 0 if target_tile == "T31TFJ" else 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else 3
                            elif "joint" and "BurnScars" in Target_region:
                                region_idx = 0 if target_tile == "T11SMT" else 1 if target_tile == "T16REV" else None
                            if "replace_final_block" in model_type:
                                _, pred = model(
                                    delta_days,
                                    x,
                                    y,
                                    torch.tensor([region_indx], device=device),
                            )
                            elif "unet" in model_type or "micro_unet" in model_type:
                                pred = model(s2data)

                            pred = getattr(pred, "output", pred)

                            if pred.dim() == 3:
                                pred = pred.unsqueeze(1)

                            metrictracker.update(pred.squeeze(1), label)

                    results = metrictracker.compute()

                    row = {
                        "target_region": Target_region,
                        "val_fold": val_fold,
                        "source_region": source_region,
                        "seed_or_run": run_name,
                        "checkpoint_path": ckpt_path,
                    }

                    for metric_name, metric_value in results.items():
                        row[metric_name] = float(metric_value.detach().cpu())

                    results_rows.append(row)

                    del model
                    del metrictracker
                    torch.cuda.empty_cache()

            del dataloader
            del val_dataset
            torch.cuda.empty_cache()


    df_results = pd.DataFrame(results_rows)

    df_results.to_csv(
        "{}_PASTIS_results_all_target_regions.csv".format(model_type),
        index=False,
    )

    df_results

Building val image label pairs: 100%|██████████| 18/18 [00:49<00:00,  2.74s/it]


Found 2736 samples for val


Building val image label pairs: 100%|██████████| 18/18 [00:51<00:00,  2.87s/it]                                            


Found 2610 samples for val


Building val image label pairs: 100%|██████████| 18/18 [00:48<00:00,  2.72s/it]                                            


...

Found 1470 samples for val


Building val image label pairs: 100%|██████████| 15/15 [00:28<00:00,  1.87s/it]                                          


Found 1665 samples for val


Building val image label pairs: 100%|██████████| 15/15 [00:28<00:00,  1.88s/it]                                            


Found 1845 samples for val


In [4]:
df_results

,target_region,val_fold,source_region,seed_or_run,checkpoint_path,accuracy_macro,accuracy_micro,f1_macro,f1_micro,jaccard_macro,jaccard_micro,precision_macro,precision_micro,recall_macro,recall_micro
0,PASTIS_local_T31TFM,1,PASTIS_local_T32ULU,2026-05-14_08-46-47,/home/user/results_local/finetuning_results_PA...,0.238315,0.717537,0.250168,0.717537,0.180134,0.559499,0.371920,0.717537,0.238315,0.717537
1,PASTIS_local_T31TFM,1,PASTIS_local_T32ULU,2026-05-14_08-51-43,/home/user/results_local/finetuning_results_PA...,0.233128,0.716995,0.243934,0.716995,0.176304,0.558841,0.358988,0.716995,0.233128,0.716995
2,PASTIS_local_T31TFM,1,PASTIS_local_T32ULU,2026-05-14_08-56-48,/home/user/results_local/finetuning_results_PA...,0.239022,0.721194,0.252988,0.721194,0.182429,0.563959,0.389209,0.721194,0.239022,0.721194
3,PASTIS_local_T31TFM,1,PASTIS_local_T32ULU,2026-05-14_09-01-41,/home/user/results_local/finetuning_results_PA...,0.232400,0.710362,0.236953,0.710362,0.171820,0.550822,0.338044,0.710362,0.232400,0.710362
4,PASTIS_local_T31TFM,1,PASTIS_local_T32ULU,2026-05-14_09-06-38,/home/user/results_local/finetuning_results_PA...,0.227074,0.705563,0.233487,0.705563,0.168367,0.545073,0.353978,0.705563,0.227074,0.705563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,PASTIS_local_T30UXV,5,PASTIS_local_T31TFJ,2026-05-14_11-13-52,/home/user/results_local/finetuning_results_PA...,0.096100,0.328957,0.078317,0.328957,0.049588,0.196857,0.173920,0.328957,0.096100,0.328957
296,PASTIS_local_T30UXV,5,PASTIS_local_T31TFJ,2026-05-14_11-20-54,/home/user/results_local/finetuning_results_PA...,0.097287,0.315360,0.063866,0.315360,0.040957,0.187197,0.176847,0.315360,0.097287,0.315360
297,PASTIS_local_T30UXV,5,PASTIS_local_T31TFJ,2026-05-14_11-28-01,/home/user/results_local/finetuning_results_PA...,0.142571,0.356380,0.090850,0.356380,0.058414,0.216827,0.186113,0.356380,0.142571,0.356380
298,PASTIS_local_T30UXV,5,PASTIS_local_T31TFJ,2026-05-14_11-35-04,/home/user/results_local/finetuning_results_PA...,0.116360,0.349271,0.080482,0.349271,0.051887,0.211586,0.175578,0.349271,0.116360,0.349271


In [ ]:
model_type_list = ["unet"]

avg_results = {}

for model_type in model_type_list:
    csv_path = f"{model_type}_PASTIS_results_all_target_regions.csv"

    print(f"\n========== {model_type} ==========")

    df_results = pd.read_csv(csv_path)
    
    df_results["training_type"] = (
        df_results["target_region"]
        .str.extract(r"_(joint|local)_")
    )
    metadata_cols = [
        "model_type",
        "target_region",
        "val_fold",
        "source_region",
        "seed_or_run",
        "checkpoint_path",
        "training_type",
    ]

    metric_cols = [
        col for col in df_results.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    # Average over seeds/runs
    df_seed_avg = (
        df_results
        .groupby(["training_type", "target_region", "val_fold", "source_region"], as_index=False)[metric_cols]
        .mean()
    )

    # Average over validation folds
    df_fold_avg = (
        df_seed_avg
        .groupby(["training_type", "target_region", "source_region"], as_index=False)[metric_cols]
        .mean()
    )

    # One result per target region
    df_joint_local_avg = (
        df_fold_avg
        .groupby("training_type", as_index=False)[metric_cols]
        .mean()
    )

    avg_results[model_type] = df_joint_local_avg

    print(df_joint_local_avg)


========== unet ==========
  training_type  accuracy_macro  accuracy_micro  f1_macro  f1_micro  \
0         local        0.179635        0.571934  0.155727  0.571934   

   jaccard_macro  jaccard_micro  precision_macro  precision_micro  \
0       0.107674        0.40755         0.225564         0.571934   

   recall_macro  recall_micro  
0      0.179635      0.571934  
